***

*Course:* [Math 535](https://people.math.wisc.edu/~roch/mmids/) - Mathematical Methods in Data Science (MMiDS)  
*Chapter:* 3-Singular value decomposition   
*Author:* [Sebastien Roch](https://people.math.wisc.edu/~roch/), Department of Mathematics, University of Wisconsin-Madison  
*Updated:* Jan 27, 2024   
*Copyright:* &copy; 2024 Sebastien Roch

***

In [ ]:
# FOR e-BOOK ONLY
import os, sys
sys.path.insert(0, os.path.abspath('../../utils')) # use directory to mmids.py

In [ ]:
# PYTHON 3
import numpy as np
from numpy import linalg as LA
import matplotlib.pyplot as plt
import pandas as pd
import mmids
seed = 535
rng = np.random.default_rng(seed)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# FOR TeX ONLY
#plt.rcParams['figure.figsize'] = [4.00, 2.25] # for high-def figs
#plt.rcParams['figure.dpi'] = 200 # for high-def figs

$\newcommand{\horz}{\rule[.5ex]{2.5ex}{0.5pt}}$

## Power iteration

There is in general [no exact method](https://math.stackexchange.com/questions/2582300/what-does-the-author-mean-by-no-method-exists-for-exactly-computing-the-eigenva) for computing SVDs. Instead we must rely on iterative methods, that is, methods that progressively approach the solution. We describe in this section the power iteration method. This method is behind an effective numerical approach for computing SVDs.

The focus here is on numerical methods and we will not spend much time computing SVDs by hand. But we start with a connection between SVDs and a certain eigendecomposition which can be used for this purpose on small examples.

### Connection between the spectral decomposition and the singular value decomposition

In previous sections, we saw that the proof of the spectral theorem tracks closely that of the existence of the SVD. Here we make the connection between the two explicit by showing how to compute the SVD using an appropriate spectral decomposition.

One might hope that the SVD of a symmetric matrix generates identical left and right singular vectors. However that is not the case.

**EXAMPLE:** It can be shown that an SVD of $A = (-1)$ is $A = (1)\,(1)\,(-1)$. That is, $\mathbf{u}_1 = (1)$ and $\mathbf{v} = (-1)$. $\lhd$

*The construction:* Let $A \in \mathbb{R}^{m \times n}$ and note that $A^T A$ is symmetric. Hence the latter has a spectral decomposition

$$
A^T A = Q \Lambda Q^T.
$$

Order the eigenvalues in decreasing order of absolute value $\lambda_1, \ldots, \lambda_n$ with $|\lambda_1| \geq \cdots \geq |\lambda_n|$. In particular, assume that the eigenvalues $\lambda_1,\ldots,\lambda_r$ are nonzero while $\lambda_{r+1} = \cdots = \lambda_n = 0$. Let $\mathbf{q}_1,\ldots,\mathbf{q}_n$ be the corresponding eigenvectors. Let $Q_1 \in \mathbb{R}^{n \times r}$ be the matrix whose columns are $\mathbf{q}_1,\ldots,\mathbf{q}_r$ and $\Lambda_1 \in \mathbb{R}^{r \times r}$ be the diagonal matrix with $\lambda_1,\ldots,\lambda_r$ on its diagonal. Similarly, let $Q_2 \in \mathbb{R}^{n \times (n-r)}$ be the matrix whose columns are $\mathbf{q}_{r+1},\ldots,\mathbf{q}_n$ and $\Lambda_2 = \mathbf{0} \in \mathbb{R}^{(n-r) \times (n-r)}$. We are now ready for our main claim.

**THEOREM** **(SVD via Spectral Decomposition)** Let $A \in \mathbb{R}^{m \times n}$ and let $Q_1, \Lambda_1$ be as above. Define

$$
U = A Q_1 \Lambda_1^{-1/2} \quad \text{and} \quad \Sigma = \Lambda_1^{1/2} \quad \text{and} \quad V = Q_1.
$$

Then $A = U \Sigma V^T$ is a singular value decomposition of $A$. $\sharp$

*Proof idea:* Check by hand that all properties of the SVD are satisfied by the construction above.

*Proof:* The matrix $\Sigma$ is diagonal and, by construction, the columns of $V$ are orthonormal. Because $A^T A$ is positive semidefinite, the eignevalues are non-negative. So it remains to prove two things: that $A = U \Sigma V^T$ and that the columns of $U$ are orthonormal.

**LEMMA** **(Step 1)** It holds that $A = U \Sigma V^T$. $\flat$

*Proof:* By direct computation, we have

$$
U \Sigma V^T
= A Q_1 \Lambda_1^{-1/2} \Lambda_1^{1/2} Q_1^T
= A Q_1 Q_1^T.
$$

The matrix $Q_1 Q_1^T$ is an orthogonal projection on the subspace spanned by the vectors $\mathbf{q}_1,\ldots,\mathbf{q}_r$. Similarly, the matrix $Q_2 Q_2^T$ is an orthogonal projection on the orthogonal complement (spanned by $\mathbf{q}_{r+1},\ldots,\mathbf{q}_n$). Hence $Q_1 Q_1^T + Q_2 Q_2^T = I_{n \times n}$. Replacing above we get

$$
U \Sigma V^T
= A (I_{n \times n} - Q_2 Q_2^T)
= A - A Q_2 Q_2^T.
$$

Now note that for any $\mathbf{q}_i$, $i=r+1,\ldots,n$, we have $A^T A \mathbf{q}_i = \mathbf{0}$, so that $\mathbf{q}_i^T A^T A \mathbf{q}_i = \|A \mathbf{q}_i\|^2 = 0$. That implies that $A \mathbf{q}_i = \mathbf{0}$ and further $A Q_2 = \mathbf{0}$. Substituting above concludes the proof. $\square$

**LEMMA** **(Step 2)** The columns of $U$ are orthonormal. $\flat$

*Proof:* By direct computation,

$$
U^T U 
= (A Q_1 \Lambda_1^{-1/2})^T A Q_1 \Lambda_1^{-1/2}
= \Lambda_1^{-1/2} Q_1^T A^T A Q_1 \Lambda_1^{-1/2}.
$$

Because the columns of $Q_1$ are eigenvectors of $A^T A$, we have that $A^T A Q_1 = Q_1 \Lambda_1$. Further those eigenvectors are orthonormal so that $Q_1^T Q_1 = I_{r \times r}$. Plugging above and simplifying gives

$$
\Lambda_1^{-1/2} Q_1^T A^T A Q_1 \Lambda_1^{-1/2}
= \Lambda_1^{-1/2} Q_1^T Q_1 \Lambda_1 \Lambda_1^{-1/2}
= \Lambda_1^{-1/2} I_{r \times r} \Lambda_1 \Lambda_1^{-1/2}
= I_{r \times r},
$$

as claimed. $\square$

That concludes the proof of the theorem. $\square$

### Power iteration lemma

We now derive the main idea behind an algorithm to compute eigenvectors and singular vectors.

**Eigenvectors in semidefinite case** We start with eigenvectors. For our purposes, it will suffice to (1) restrict ourselves to positive semidefinite matrices and (2) focus on the largest eigenvalue.

Let $A$ be a symmetric, positive semindefinite matrix in $\mathbb{R}^{d \times d}$. By the *Spectral Theorem*, it has an eigenvector decomposition

$$
A
= Q \Lambda Q^T
= \sum_{i=1}^d \lambda_i \mathbf{q}_i \mathbf{q}_i^T
$$

where further $0 \leq \lambda_d \leq \cdots \leq \lambda_1$ by the *Characterization of Positive Semidefiniteness*.  To see how one might go about computing it, we start with the following observation. 

Because of the orthogonality of $Q$, the powers of $A$ have a simple representation. The square gives

$$
A^2 
= (Q \Lambda Q^T) (Q \Lambda Q^T)
= Q \Lambda^2 Q^T.
$$

Repeating

$$
A^{k}
= Q \Lambda^{k} Q^T.
$$

Further

$$
\Lambda^k
= \begin{pmatrix}
\lambda_1^{k} & 0 & \cdots & 0\\
0 & \lambda_2^{k} & \cdots & 0\\
\vdots & \vdots & \ddots & \vdots\\
0 & 0 & \cdots & \lambda_d^{k}
\end{pmatrix}.
$$

When $\lambda_1 > \lambda_2, \ldots, \lambda_d$, which is often the case with real datasets, we get that $\lambda_1^{k} \gg \lambda_2^{k}, \ldots, \lambda_d^{k}$ when $k$ is large. Then, we get the approximation

$$
A^{k}
=
\sum_{i=1}^d \lambda_i^{k} \mathbf{q}_i \mathbf{q}_i^T
\approx
\lambda_1^{k} \mathbf{q}_1 \mathbf{q}_1^T.
$$

This leads to the following idea to compute $\mathbf{q}_1$:

**LEMMA** **(Power Iteration in the Positive Semidefinite Case)** Let $A$ be a symmetric, positive semindefinite matrix in $\mathbb{R}^{d \times d}$ with eigenvector decomposition $A= Q \Lambda Q^T$ where the eigenvalues satisfy $0 \leq \lambda_d \leq \cdots \leq \lambda_2 < \lambda_1$. Assume that $\mathbf{x} \in \mathbb{R}^d$ is a vector such that $\langle \mathbf{q}_1, \mathbf{x} \rangle > 0$. Then

$$
\frac{A^{k} \mathbf{x}}{\|A^{k} \mathbf{x}\|} \to \mathbf{q}_1
$$

as $k \to +\infty$. If instead $\langle \mathbf{q}_1, \mathbf{x} \rangle < 0$, then the limit is $- \mathbf{q}_1$. $\flat$

The proof is similar to the case of singular vectors, which is detailed next.

**SVD** The same approach can be used to compute singular vectors. Let $U \Sigma V^T$ be a (compact) SVD of $A$. To see how one might go about computing it, we start with the following observation. Now we can mimick the calculations that led us to the *Power Iteration Lemma* in the eigenvector case. Because of the orthogonality of $U$ and $V$, the powers of $A^T A$ have a simple representation. Indeed

$$
B = A^T A
= (U \Sigma V^T)^T (U \Sigma V^T)
= V \Sigma^T U^T U \Sigma V^T
= V \Sigma^T \Sigma V^T,
$$

so that

$$
B^2 
= (V \Sigma^T \Sigma V^T) (V \Sigma^T \Sigma V^T)
= V (\Sigma^T \Sigma)^2 V^T.
$$

Repeating

$$
B^{k}
= V (\Sigma^T \Sigma)^{k} V^T.
$$

Further, 

$$
\widetilde{\Sigma}
= \Sigma^T \Sigma
= \begin{pmatrix}
\sigma_1^2 & 0 & \cdots & 0\\
0 & \sigma_2^2 & \cdots & 0\\
\vdots & \vdots & \ddots & \vdots\\
0 & 0 & \cdots & \sigma_r^2
\end{pmatrix}.
$$

So

$$
\widetilde{\Sigma}^k
= \begin{pmatrix}
\sigma_1^{2k} & 0 & \cdots & 0\\
0 & \sigma_2^{2k} & \cdots & 0\\
\vdots & \vdots & \ddots & \vdots\\
0 & 0 & \cdots & \sigma_r^{2k}
\end{pmatrix}.
$$

When $\sigma_1 > \sigma_2, \ldots, \sigma_r$, which -- again -- is often the case with real datasets, we get that $\sigma_1^{2k} \gg \sigma_2^{2k}, \ldots, \sigma_r^{2k}$ when $k$ is large. Then, we get the approximation

$$
B^{k}
=
\sum_{j=1}^r \sigma_j^{2k} \mathbf{v}_j \mathbf{v}_j^T
\approx
\sigma_1^{2k} \mathbf{v}_1 \mathbf{v}_1^T.
$$

Finally, we arrive at:

**LEMMA** **(Power Iteration)** Let $A \in \mathbb{R}^{n\times m}$ be a matrix and let $U \Sigma V^T$ be a (compact) SVD of $A$ such that $\sigma_1 > \sigma_2 > 0$. Define $B = A^T A$ and assume that $\mathbf{x} \in \mathbb{R}^m$ is a vector satisfying $\langle \mathbf{v}_1, \mathbf{x} \rangle > 0$. Then

$$
\frac{B^{k} \mathbf{x}}{\|B^{k} \mathbf{x}\|} \to \mathbf{v}_1
$$

as $k \to +\infty$. If instead $\langle \mathbf{v}_1, \mathbf{x} \rangle < 0$, then the limit is $- \mathbf{v}_1$. $\flat$

*Proof idea:* We use the approximation above and divide by the norm to get a unit norm vector in the direction of $\mathbf{v}_1$.

*Proof:* We have

$$
B^{k}\mathbf{x}
=
\sum_{j=1}^r \sigma_j^{2k} \mathbf{v}_j \mathbf{v}_j^T \mathbf{x}.
$$

So

\begin{align*}
\frac{B^{k} \mathbf{x}}{\|B^{k} \mathbf{x}\|}
&= 
\sum_{j=1}^r \mathbf{v}_j \frac{\sigma_j^{2k} (\mathbf{v}_j^T \mathbf{x})}
{\|B^{k} \mathbf{x}\|}\\
&= 
\mathbf{v}_1 \left\{\frac{\sigma_1^{2k} (\mathbf{v}_1^T \mathbf{x})}
{\|B^{k} \mathbf{x}\|}\right\}
+ \sum_{j=2}^r \mathbf{v}_j \left\{\frac{\sigma_j^{2k} (\mathbf{v}_j^T \mathbf{x})}
{\|B^{k} \mathbf{x}\|}\right\}.
\end{align*}

This goes to $\mathbf{v}_1$ as $k\to +\infty$ if the expression in the first curly brackets goes to $1$ and the one in the second curly brackets goes to $0$. We prove this in the next claim.

**LEMMA** As $k\to +\infty$,

$$
\frac{\sigma_1^{2k} (\mathbf{v}_1^T \mathbf{x})}
{\|B^{k} \mathbf{x}\|} \to 1
\qquad
\text{and}
\qquad
\frac{\sigma_j^{2k} (\mathbf{v}_j^T \mathbf{x})}
{\|B^{k} \mathbf{x}\|} \to 0, 
\ 
j = 2,\ldots,r.
$$

$\flat$

*Proof:* Because the $\mathbf{v}_j$'s are an orthonormal basis,

$$
\|B^{k}\mathbf{x}\|^2
= 
\sum_{j=1}^r \left[\sigma_j^{2k} (\mathbf{v}_j^T \mathbf{x})\right]^2
=
\sum_{j=1}^r \sigma_j^{4k} (\mathbf{v}_j^T \mathbf{x})^2.
$$

So, as $k\to +\infty$, using the fact that $\langle \mathbf{v}_1, \mathbf{x} \rangle \neq 0$ by assumption

\begin{align*}
\frac{\|B^{k}\mathbf{x}\|^2}{\sigma_1^{4k} (\mathbf{v}_1^T \mathbf{x})^2}
&=
1 + \sum_{j=2}^r \frac{\sigma_j^{4k} (\mathbf{v}_j^T \mathbf{x})^2}{\sigma_1^{4k} (\mathbf{v}_1^T \mathbf{x})^2}\\
&=
1 + \sum_{j=2}^r \left(\frac{\sigma_j}{\sigma_1}\right)^{4k} \frac{(\mathbf{v}_j^T \mathbf{x})^2}{(\mathbf{v}_1^T \mathbf{x})^2}\\
&\to 
1,
\end{align*}

since $\sigma_j < \sigma_1$ for all $j =2,\ldots,r$. That implies the first part of the claim by taking a square root and using $\langle \mathbf{v}_1, \mathbf{x} \rangle > 0$. The second part of the claim follows essentially from the same argument. $\square$ $\square$

**EXAMPLE:** Let's revisit the example

$$
A = \begin{pmatrix}
1 & 0\\
-1 & 0
\end{pmatrix}.
$$

We previously compute its SVD and found that 

$$
\mathbf{v}_1 
= \begin{pmatrix}
1\\
0
\end{pmatrix}.
$$



This time we use the *Power Iteration Lemma*. Here

$$
B = A^T A 
= \begin{pmatrix}
2 & 0\\
0 & 0
\end{pmatrix}.
$$

Taking powers of this matrix is easy

$$
B^k = \begin{pmatrix}
2^k & 0\\
0 & 0
\end{pmatrix}.
$$

Let's choose an arbitrary initial vector $\mathbf{x}$, say $(-1, 2)^T$. Then

$$
B^k \mathbf{x}
= \begin{pmatrix}
-2^k\\
0
\end{pmatrix}
\quad
\text{and}
\quad
\|B^k \mathbf{x}\| = 2^k.
$$

So

$$
\frac{B^{k} \mathbf{x}}{\|B^{k} \mathbf{x}\|} \to \begin{pmatrix}
-1\\
0
\end{pmatrix} 
=
- \mathbf{v}_1,
$$

as $k \to +\infty$. In fact, in this case, converges occurs after one step. $\lhd$

### Computing the top singular vector

Power iteration gives us a way to compute $\mathbf{v}_1$ -- at least approximately if we use a large enough $k$. But how do we find an appropriate vector $\mathbf{x}$, as required by the *Power Iteration Lemma*? It turns out that a random vector will do. For instance, let $\mathbf{X}$ be an $m$-dimensional spherical Gaussian with mean $0$ and variance $1$. Then, $\mathbb{P}[\langle \mathbf{v}_1, \mathbf{X} \rangle = 0] = 0$. We will show this when we discuss multivariate Gaussians. Note that, if $\langle \mathbf{v}_1, \mathbf{X} \rangle < 0$, we will instead converge to $-\mathbf{v}_1$ which is also a right singular vector.


**NUMERICAL CORNER:** We implement the algorithm suggested by the *Power Iteration Lemma*. That is, we compute $B^{k} \mathbf{x}$, then normalize it. To obtain the corresponding singular value and left singular vector, we use that $\sigma_1 = \|A \mathbf{v}_1\|$ and $\mathbf{u}_1 = A \mathbf{v}_1/\sigma_1$.

In [ ]:
def topsing(A, maxiter=10):
    x = rng.normal(0,1,np.shape(A)[1])
    B = A.T @ A
    for _ in range(maxiter):
        x = B @ x
    v = x / LA.norm(x)
    s = LA.norm(A @ v)
    u = A @ v / s
    return u, s, v

We will apply it to our previous two-cluster example. The necessary functions are in [mmids.py](https://raw.githubusercontent.com/MMiDS-textbook/MMiDS-textbook.github.io/main/utils/mmids.py), which is available on the [GitHub of the notes](https://github.com/MMiDS-textbook/MMiDS-textbook.github.io/tree/main). 

In [ ]:
d, n, w = 10, 100, 3.
X1, X2 = mmids.two_clusters(d, n, w)
X = np.concatenate((X1, X2), axis=0)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111,aspect='equal')
ax.scatter(X[:,0], X[:,1])
plt.show()

Let's compute the top singular vector.

In [ ]:
u, s, v = topsing(X)
print(v)

This is approximately $-\mathbf{e}_1$. We get roughly the same answer (possibly up to sign) from Python's [`numpy.linalg.svd`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html) function.

In [ ]:
u, s, vh = LA.svd(X)
print(vh.T[:,0])

Recall that, when we applied $k$-means clustering to this example with $d=1000$ dimension, we obtained a very poor clustering. Let's try again after projecting onto the top singular vector.

In [ ]:
d, n, w = 1000, 100, 3.
X1, X2 = mmids.two_clusters(d, n, w)
X = np.concatenate((X1, X2), axis=0)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111,aspect='equal')
ax.scatter(X[:,0], X[:,1])
plt.show()

In [ ]:
assign = mmids.kmeans(X, 2)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111,aspect='equal')
ax.scatter(X[:,0], X[:,1], c=assign)
plt.show()

Let's try again, but after projecting on the top singular vector. Recall that this corresponds to finding the best one-dimensional approximating subspace. The projection can be computed using the truncated SVD $Z= U_{(1)} \Sigma_{(1)} V_{(1)}^T$. We can interpret the rows of $U_{(1)} \Sigma_{(1)}$ as the coefficients of each data point in the basis $\mathbf{v}_1$. We will work in that basis. We need one small hack: because our implementation of $k$-means clustering expects data points in at least $2$ dimension, we add a column of $0$'s.

In [ ]:
u, s, v = topsing(X)
Xproj = np.stack((u*s, np.zeros(np.shape(X)[0])), axis=-1)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111,aspect='equal')
ax.scatter(Xproj[:,0], Xproj[:,1])
plt.ylim([-3,3])
plt.show()

There is a small - yet noticeable - gap around 0.

A histogram of the first component of `Xproj` gives a better sense of the density of points.

In [ ]:
plt.hist(Xproj[:,0])
plt.show()

We run $k$-means clustering on the projected data.

In [ ]:
assign = mmids.kmeans(Xproj, 2)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111,aspect='equal')
ax.scatter(X[:,0], X[:,1], c=assign)
plt.show()

Much better. We will give an explanation of this outcome in an upcoming (optional) subsection. In essence, quoting [BHK, Section 7.5.1]:

> [...] let's understand the central advantage of doing the projection to [the top $k$ right singular vectors]. It is simply that for any reasonable (unknown) clustering of data points, the projection brings data points closer to their cluster centers.

Finally, looking at the top right singular vector (or its first ten entries for lack of space), we see that it does align quite well (but not perfectly) with the first dimension. In the next (optional) section, we try again with the top two singular vectors.

In [ ]:
print(v[:10])

$\unlhd$

### Computing more singular vectors

We have shown how to compute the first singular vector. How do we compute more singular vectors? One approach is to first compute $\mathbf{v}_1$ (or $-\mathbf{v}_1$), then find a vector $\mathbf{y}$ orthogonal to it, and proceed as above. And then we repeat until we have all $m$ right singular vectors.

We are often interested only in the top, say $\ell < m$, singular vectors. An alternative approach in that case is to start with $\ell$ random vectors and, first, find an orthonormal basis for the space they span. Then to quote [BHK, Section 3.7.1]:

> Then compute $B$ times each of the basis vectors, and find an orthonormal basis for the space spanned by the resulting vectors. Intuitively, one has applied $B$ to a subspace rather than a single vector. One repeatedly applies $B$ to the subspace, calculating an orthonormal basis after each application to prevent the subspace collapsing to the one dimensional subspace spanned by the first singular vector.

We will not prove here that this approach, known as orthogonal iteration, works. The proof is similar to that of the *Power Iteration Lemma*.

**NUMERICAL CORNER:** We implement this last algorithm. We will need our previous implementation of *Gram-Schimdt*.

In [ ]:
def svd(A, l, maxiter=100):
    V = rng.normal(0,1,(np.size(A,1),l))
    for _ in range(maxiter):
        W = A @ V
        Z = A.T @ W
        V, R = mmids.gramschmidt(Z)
    W = A @ V
    S = [LA.norm(W[:, i]) for i in range(np.size(W,1))]
    U = np.stack([W[:,i]/S[i] for i in range(np.size(W,1))],axis=-1)
    return U, S, V

Note that above we avoided forming the matrix $A^T A$. With a small number of iterations, that approach potentially requires fewer arithmetic operations overall and it allows to take advantage of the possible sparsity of $A$ (i.e. the fact that it may have many zeros).

We apply it again to our two-cluster example.

In [ ]:
d, n, w = 1000, 100, 3.
X1, X2 = mmids.two_clusters(d, n, w)
X = np.concatenate((X1, X2), axis=0)

Let's try again, but after projecting on the top two singular vectors. Recall that this corresponds to finding the best two-dimensional approximating subspace. The projection can be computed using the truncated SVD $Z= U_{(2)} \Sigma_{(2)} V_{(2)}^T$. We can interpret the rows of $U_{(2)} \Sigma_{(2)}$ as the coefficients of each data point in the basis $\mathbf{v}_1,\mathbf{v}_2$. We will work in that basis.

In [ ]:
U, S, V = svd(X, 2)
Xproj = np.stack((U[:,0]*S[0], U[:,1]*S[1]), axis=-1)
assign = mmids.kmeans(Xproj, 2)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111,aspect='equal')
ax.scatter(X[:,0], X[:,1], c=assign)
plt.show()

Finally, looking at the first two right singular vectors, we see that the first one does align quite well with the first dimension.

In [ ]:
print(np.stack((V[:,0], V[:,1]), axis=-1))

$\unlhd$